In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp
import ROOT
from ROOT import TMVA

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
import analysis_village.cc1pi.DataFrameUtils.DFCleaning as DFUtils
import analysis_village.cc1pi.CutMasks.CutMasks as CutMasks

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh



In [ ]:
## Check keys in each file
optimization_file = "/scratch/7DayLifetime/lpelegrina/cc1pi_op_1e20.df"
development_sample_file = "/scratch/7DayLifetime/lpelegrina/cc1pi_op.df"
print("keys in test_file")
splh.print_keys(optimization_file)

## Check split multiplicity
print("mc_bnb_cosmic_file n_split: %d" %splh.get_n_split(optimization_file))

In [ ]:
## Define keys to load
print('MC dataframes')
n_max_concat = 10 ## for big files, each key could have more than one split
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
optimization_df = splh.load_dfs(optimization_file, keys2load, n_max_concat)
development_sample_df = splh.load_dfs(development_sample_file, keys2load, n_max_concat)
print('test data loaded!')

In [ ]:
#Perform duplication validation
print("duplication for Spring Production BNB + Cosmic sample")
DFUtils.find_duplicate_run_evt_combinations(optimization_df['hdr'])
DFUtils.find_duplicate_run_evt_combinations(development_sample_df['hdr'])


In [ ]:
DFUtils.plot_duplicate_run_subrun_evt_distribution(optimization_df["hdr"], "test_df")
DFUtils.plot_duplicate_run_subrun_evt_distribution(development_sample_df["hdr"], "dev_sample_df")

In [ ]:
### Filter the hdr DataFrame first, then filter other DataFrames by matching with the hdr DataFrame
optimization_df["hdr"] = DFUtils.filter_unique_events(optimization_df["hdr"])
DFUtils.find_duplicate_run_evt_combinations(optimization_df["hdr"])
#filter the rest of dataframe keys
for key in keys2load:
    if key == "hdr":
        continue
    optimization_df[key] = DFUtils.filter_using_hdr(optimization_df[key], optimization_df["hdr"])


development_sample_df["hdr"] = DFUtils.filter_unique_events(development_sample_df["hdr"])
DFUtils.find_duplicate_run_evt_combinations(development_sample_df["hdr"])
#filter the rest of dataframe keys
for key in keys2load:
    if key == "hdr":
        continue
    development_sample_df[key] = DFUtils.filter_using_hdr(development_sample_df[key], development_sample_df["hdr"])    

In [ ]:
SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]

In [ ]:
def get_n_evt(df):
    unique_count = df.index.droplevel(
        list(df.index.names[2:])  # drop everything except first two levels
    ).nunique()
    return unique_count

In [ ]:
## Collect pot scale for MC
mc_tot_pot = optimization_df["hdr"]['pot'].sum()
#mc_low_th_tot_pot = mc_rockbox_th1to100_dfs["hdr"]['pot'].sum()

data_tot_pot = 3.63462e+18
#data_tot_pot = data_bnb_light_dfs["hdr"]['pot'].sum()
#data_tot_TOR860 = data_bnb_light_dfs["pot"]['TOR860'].sum()
#data_tot_TOR875 = data_bnb_light_dfs["pot"]['TOR875'].sum()

print("mc_tot_pot: %e" %(mc_tot_pot))
#print("mc_low_thtot_pot: %e" %(mc_low_th_tot_pot))

#print("data_tot_pot: %e" %(data_tot_pot))
#print("data_tot_TOR860: %e" %(data_tot_TOR860))
#print("data_tot_TOR875: %e" %(data_tot_TOR875))

target_pot = data_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
#mc_low_th_scale = target_pot / mc_low_th_tot_pot
print("MC POT scale: %.3f" %(mc_pot_scale))
#print("MC Low Th. POT scale: %.3f" %(mc_low_th_scale))

In [ ]:
## Comparison between observed and expected total number of recorded spills
n_evt_mc = get_n_evt(optimization_df["hdr"])
#n_evt_mc_low_th = get_n_evt(mc_rockbox_th1to100_dfs["hdr"])

#print("n_evt_data_onbeam: %d" %n_record_spill_data)
#print("n_evt_exp.: %f" %(n_evt_mc * mc_pot_scale + n_evt_mc_low_th * mc_low_th_scale +n_record_spill_offbeam_data * intime_gate_scale))
print("- n_evt_mc: %f" %(n_evt_mc * mc_pot_scale))
#print("- n_evt_mc_low_th: %f" %(n_evt_mc_low_th * mc_low_th_scale))
#print("- n_evt_data_offbeam: %f" %(n_record_spill_offbeam_data * intime_gate_scale))

In [ ]:
evt_df = optimization_df['cc1pi']
hdr_df = optimization_df['hdr']
nu_df = optimization_df['nudf']

dev_sample_evt_df = development_sample_df['cc1pi']
dev_sample_nu_df = development_sample_df['nudf']

In [ ]:
new_columns = []
for c in nu_df.columns:
    new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
nu_df.columns = pd.MultiIndex.from_tuples(new_columns)

new_columns = []
for c in dev_sample_nu_df.columns:
    new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
dev_sample_nu_df.columns = pd.MultiIndex.from_tuples(new_columns)

In [ ]:
matchdf = ph.multicol_merge(evt_df.reset_index(), nu_df.reset_index(),
                            left_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("slc", "tmatch","idx","","","")],
                            right_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("rec.mc.nu..index", "","","","","")], 
                            how="left") ## -- save all sllices
#Reindex so it is again "__ntuple","entry", "slice_id"
matchdf = matchdf.set_index(evt_df.index.names, verify_integrity=True)
#Remove "rec.mc.nu..index"
matchdf = matchdf.drop(columns=[('rec.mc.nu..index','','','','','')])
matchdf.loc[:, ('truth', 'nu_categ','','','','')] = (
    matchdf.loc[:, ('truth', 'nu_categ','','','','')].fillna('cosmic')
)

dev_sample_matchdf = ph.multicol_merge(dev_sample_evt_df.reset_index(), dev_sample_nu_df.reset_index(),
                            left_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("slc", "tmatch","idx","","","")],
                            right_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("rec.mc.nu..index", "","","","","")], 
                            how="left") ## -- save all sllices
#Reindex so it is again "__ntuple","entry", "slice_id"
dev_sample_matchdf = dev_sample_matchdf.set_index(dev_sample_evt_df.index.names, verify_integrity=True)
#Remove "rec.mc.nu..index"
dev_sample_matchdf = dev_sample_matchdf.drop(columns=[('rec.mc.nu..index','','','','','')])
dev_sample_matchdf.loc[:, ('truth', 'nu_categ','','','','')] = (
    dev_sample_matchdf.loc[:, ('truth', 'nu_categ','','','','')].fillna('cosmic')
)

In [ ]:
evt_df = matchdf
dev_sample_evt_df = dev_sample_matchdf

In [ ]:
def bdt_quality_mask(df, columns):
    mask = np.ones(len(df), dtype=bool)

    for col in columns:
        mask &= df[col].notna()
        mask &= df[col] >= 0

    return mask

In [ ]:
chi2_p_cut = 80
chi2_mu_cut = 20
len_cut = 10
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')

col_chi2_exp_pol  = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_chi2_exp_pol_3var = ('pfp', 'trk', 'chi2_exp_pol_3var', '', '', '')
col_frac_50 = ('pfp', 'trk', 'frac50', '', '', '')

#BDT_columns = [col_chi2_mu, col_chi2_p, col_len, col_chi2_exp_pol, col_chi2_exp_pol_3var, col_frac_50]
BDT_columns = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_frac_50]

cut_mask = CutMasks.nu_score_cut_mask(evt_df)  & CutMasks.track_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(evt_df,SLICE_LEVELS)

#Define the track df
signal_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[
    cut_mask & 
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 2212) ) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for training
bkg_df = bkg_df[bkg_df.pfp.is_exiting == False]
signal_df = signal_df[signal_df.pfp.is_exiting == False]
print(len(signal_df))
print(len(bkg_df))

bdt_mask_signal = bdt_quality_mask(signal_df, BDT_columns)
bdt_mask_bkg    = bdt_quality_mask(bkg_df, BDT_columns)
signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]
signal_df = signal_df[(signal_df[col_chi2_mu] < chi2_mu_cut) & (signal_df[col_chi2_p] > chi2_p_cut) & (signal_df[col_len] > len_cut)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < chi2_mu_cut) & (bkg_df[col_chi2_p] > chi2_p_cut) & (bkg_df[col_len] > len_cut)]


In [ ]:
len(bkg_df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from scipy.stats import ks_2samp
import pickle

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier, RandomForestClassifier

def create_models_for_columns(input_cols):
    """
    Create all the standard models (BDT, BDTG, BDTB, RF) for a given set of input columns.

    Parameters
    ----------
    input_cols : list of tuples
        List of MultiIndex columns to use as features.
    prefix : str
        Prefix to use in model names.

    Returns
    -------
    model_vec : dict
        Dictionary of models, keys are names like 'BDT_<prefix>', 'BDTG_<prefix>', etc.
    """
    model_vec = {}

    # --- BDT (AdaBoost) ---
    model_vec[f"BDT"] = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=3,
            min_samples_leaf=0.025,
            criterion="gini"
        ),
        n_estimators=850,
        learning_rate=0.5,
        random_state=42
    )

    # --- BDTG (Gradient Boost) ---
    model_vec[f"BDTG"] = GradientBoostingClassifier(
        n_estimators=850,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.5,
        min_samples_leaf=0.025,
        random_state=42
    )

    # --- BDTB (Bagging) ---
    model_vec[f"BDTB"] = BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=3,
            min_samples_leaf=0.025,
            criterion="gini"
        ),
        n_estimators=400,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    # --- Random Forest ---
    model_vec[f"RF"] = RandomForestClassifier(
        n_estimators=100,
        criterion='gini',
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        min_weight_fraction_leaf=0.0,
        max_features='sqrt',
        max_leaf_nodes=None,
        min_impurity_decrease=0.0,
        bootstrap=True,
        oob_score=False,
        n_jobs=-1,
        random_state=42,
        verbose=0,
        warm_start=False,
        class_weight=None,
        ccp_alpha=0.0,
        max_samples=None
    )

    return model_vec


In [ ]:
BDT_columns_normal = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_frac_50]
BDT_columns_3vars = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol_3var, col_frac_50]

model_vec_normal = create_models_for_columns(BDT_columns_normal)
model_vec_3vars = create_models_for_columns(BDT_columns_3vars)

In [ ]:
from sklearn.model_selection import train_test_split

def train_all_models(signal_df, bkg_df, input_cols, model_vec, test_size=0.2, random_state=42):
    """
    Train all models in model_vec on the provided signal/background DataFrames.

    Parameters
    ----------
    signal_df : pd.DataFrame
        DataFrame containing signal events.
    bkg_df : pd.DataFrame
        DataFrame containing background events.
    input_cols : list of tuples
        List of MultiIndex columns to use as features.
    model_vec : dict
        Dictionary of sklearn models to train. Keys are names, values are model instances.
    test_size : float
        Fraction of dataset to keep for testing.
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    trained_models : dict
        Dictionary of trained models (same keys as model_vec).
    X_train, X_test, y_train, y_test : np.ndarray
        Train/test splits used for training.
    """
    # 1️⃣ Build input arrays
    X_sig = signal_df[input_cols].values
    X_bkg = bkg_df[input_cols].values

    y_sig = np.ones(len(X_sig))
    y_bkg = np.zeros(len(X_bkg))

    X = np.vstack([X_sig, X_bkg])
    y = np.hstack([y_sig, y_bkg])

    # 2️⃣ Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    # 3️⃣ Train each model
    trained_models = {}
    for name, clf in model_vec.items():
        clf.fit(X_train, y_train)
        trained_models[name] = clf
        print(f"Trained {name}")

    return X_train, X_test, y_train, y_test


In [ ]:

# Train all models
X_train_normal, X_test_normal, y_train_normal, y_test_normal = train_all_models(
    signal_df,
    bkg_df,
    BDT_columns_normal,
    model_vec_normal,
    test_size=0.2
)

# Train all models
X_train_3vars, X_test_3vars, y_train_3vars, y_test_3vars = train_all_models(
    signal_df,
    bkg_df,
    BDT_columns_3vars,
    model_vec_3vars,
    test_size=0.2
)


In [ ]:
def get_scores(clf, X_train, X_test):
    """
    Safely get the BDT response/score for any sklearn classifier.
    Uses decision_function if available (TMVA-like), otherwise predict_proba.
    """
    # Try decision_function first (outputs values usually centered around 0)
    if hasattr(clf, "decision_function"):
        score_train = clf.decision_function(X_train)
        score_test  = clf.decision_function(X_test)
    # Fallback to predict_proba (outputs values between 0 and 1)
    else:
        score_train = clf.predict_proba(X_train)[:, 1]
        score_test  = clf.predict_proba(X_test)[:, 1]
        
    return score_train, score_test

bdt_scores_train, bdt_scores_test   = get_scores(model_vec_normal["BDT"], X_train_normal, X_test_normal)
bdt_scores_train_3vars, bdt_scores_test_3vars = get_scores(model_vec_3vars["BDT"], X_train_3vars, X_test_3vars)


bdtg_scores_train, bdtg_scores_test = get_scores(model_vec_normal["BDTG"], X_train_normal, X_test_normal)
#bdtb_scores_train, bdtb_scores_test = get_scores(model_vec_normal["BDTB"], X_train_normal, X_test_normal)
#random_forest_scores_train, random_forest_scores_test = get_scores(model_vec_normal["RF"], X_train_normal, X_test_normal)

In [ ]:
def plot_roc(y_true, scores, label):
    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC = {roc_auc:.3f})")
    return roc_auc

plt.figure(figsize=(6,6))
plot_roc(y_test_normal, bdt_scores_test, "BDT normal")
plot_roc(y_test_3vars, bdt_scores_test_3vars, "BDT 3 vars")
plt.plot([0,1],[0,1], "k--")
plt.xlabel("Background efficiency")
plt.ylabel("Signal efficiency")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(6,6))
plot_roc(y_test_normal, bdt_scores_test, "BDT")
plot_roc(y_test_normal, bdtg_scores_test, "BDTG")
#plot_roc(y_test_normal, bdtb_scores_test, "BDTB")
#plot_roc(y_test_normal, random_forest_scores_test, "RandomForest")
plt.plot([0,1],[0,1], "k--")
plt.xlabel("Background efficiency")
plt.ylabel("Signal efficiency")
plt.legend()
plt.show()

In [ ]:
def ks_test(signal_train, signal_test, bkg_train, bkg_test):
    ks_signal = ks_2samp(signal_train, signal_test).pvalue
    ks_bkg    = ks_2samp(bkg_train, bkg_test).pvalue
    return ks_signal, ks_bkg

In [ ]:
'''
sig_train = random_forest_scores_train[y_train==1]
sig_test  = random_forest_scores_test[y_test==1]
bkg_train = random_forest_scores_train[y_train==0]
bkg_test  = random_forest_scores_test[y_test==0]

'''
'''
sig_train = bdtg_scores_train[y_train_normal==1]
sig_test  = bdtg_scores_test[y_test_normal==1]
bkg_train = bdtg_scores_train[y_train_normal==0]
bkg_test  = bdtg_scores_test[y_test_normal==0]
'''

sig_train = bdt_scores_train[y_train_normal==1]
sig_test  = bdt_scores_test[y_test_normal==1]
bkg_train = bdt_scores_train[y_train_normal==0]
bkg_test  = bdt_scores_test[y_test_normal==0]

ks_s, ks_b = ks_test(sig_train, sig_test, bkg_train, bkg_test)
print("BDT KS signal p-value:", ks_s)
print("BDT KS bkg    p-value:", ks_b)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def plot_feature_importances(model, columns, title=None):
    """
    Plot feature importances for a tree-based model.
    MultiIndex column names are merged into a single string and sorted by importance.

    Parameters
    ----------
    model : sklearn estimator
        Trained tree-based model (DecisionTree, AdaBoost, GradientBoosting, RandomForest, Bagging of trees)
    columns : list of tuples
        MultiIndex columns used as input features.
    title : str, optional
        Plot title.
    """
    # 1️⃣ Get feature importances
    if hasattr(model, "estimators_") and isinstance(model, type(model)):
        try:
            # BaggingClassifier
            importances = np.mean([t.feature_importances_ for t in model.estimators_], axis=0)
        except AttributeError:
            importances = model.feature_importances_
    else:
        importances = model.feature_importances_

    # 2️⃣ Merge MultiIndex column names into a single string
    feature_names = ["_".join([str(c) for c in col if c != ""]) for col in columns]

    # 3️⃣ Create DataFrame
    df_importances = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })

    # 4️⃣ Sort by importance (descending) → most important at top
    df_importances = df_importances.sort_values(by="importance", ascending=True)  # ascending=True for horizontal bar plot

    # 5️⃣ Plot
    plt.figure(figsize=(6,4))
    plt.barh(df_importances['feature'], df_importances['importance'], color='skyblue')
    plt.xlabel("Feature importance")
    plt.ylabel("Feature")
    plt.title(title or "Feature importances")
    plt.tight_layout()
    plt.show()
    plt.show()

    return df_importances


In [ ]:
df_imp = plot_feature_importances(model_vec_normal["BDT"], BDT_columns_normal, title="BDT Feature importances")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_correlation_matrix(df, columns, title=None):
    """
    Plot correlation matrix for a list of MultiIndex columns in a DataFrame using plt.imshow.

    MultiIndex column names are merged into single strings for labeling.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the data.
    columns : list of tuples
        MultiIndex columns to compute correlation on.
    title : str, optional
        Plot title.
    """
    # 1️⃣ Select valid rows (no NaN, >=0)
    mask = np.ones(len(df), dtype=bool)
    for col in columns:
        mask &= df[col].notna()
        mask &= df[col] >= 0

    df_vars = df.loc[mask, columns]

    # 2️⃣ Merge MultiIndex names for labeling
    feature_names = ["_".join([str(c) for c in col if c != ""]) for col in columns]
    df_vars.columns = feature_names

    # 3️⃣ Compute correlation matrix
    corr = df_vars.corr().values

    # 4️⃣ Plot with plt.imshow
    plt.figure(figsize=(6,5))
    im = plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(im, label="Correlation")

    # 5️⃣ Set ticks and labels
    plt.xticks(ticks=np.arange(len(feature_names)), labels=feature_names, rotation=45, ha="right")
    plt.yticks(ticks=np.arange(len(feature_names)), labels=feature_names)

    # 6️⃣ Annotate values
    for i in range(len(feature_names)):
        for j in range(len(feature_names)):
            plt.text(j, i, f"{corr[i,j]:.2f}", ha="center", va="center", color="black")

    plt.title(title or "Correlation matrix")
    plt.tight_layout()
    plt.show()

    return corr

In [ ]:
corr_matrix = plot_correlation_matrix(signal_df, BDT_columns_normal, title="Input variable correlations")
corr_matrix = plot_correlation_matrix(bkg_df, BDT_columns_normal, title="Input variable correlations")

In [ ]:
def plot_response(sig_train, sig_test, bkg_train, bkg_test, title):
    plt.figure(figsize=(6,5))
    plt.hist(sig_train, bins=50, density=True, histtype='step', label='Signal Train', color='r')
    plt.hist(sig_test,  bins=50, density=True, histtype='step', linestyle='dashed', label='Signal Test', color='r')
    plt.hist(bkg_train, bins=50, density=True, histtype='step', label='Bkg Train', color='b')
    plt.hist(bkg_test,  bins=50, density=True, histtype='step', linestyle='dashed', label='Bkg Test', color='b')
    plt.xlabel('BDT response')
    plt.ylabel('Normalized entries')
    plt.title(title)
    plt.legend()
    plt.show()

plot_response(sig_train, sig_test, bkg_train, bkg_test, "BDT Response")

In [ ]:
with open("bdt_model_proton.pkl", "wb") as f:
    pickle.dump(model_vec_normal["BDT"], f)
'''
with open("bdtg_model.pkl", "wb") as f:
    pickle.dump(model_vec_normal["BDTG"], f)
'''
'''
with open("bdtb_model.pkl", "wb") as f:
    pickle.dump(bdtb, f)

with open("random_forest.pkl", "wb") as f:
    pickle.dump(rf, f)
'''

In [ ]:
import numpy as np
import pickle

def add_bdt_score(evt_df, bdt_model_file, input_cols, output_col):
    """
    Add a BDT score column to a MultiIndex DataFrame.

    Parameters
    ----------
    evt_df : pd.DataFrame
        The DataFrame containing the input columns.
    bdt_model_file : str
        Path to the pickle file with the trained BDT.
    input_cols : list of tuples
        List of MultiIndex columns to use as BDT inputs.
    output_col : tuple
        MultiIndex tuple for the output BDT score column.

    Returns
    -------
    pd.DataFrame
        The input DataFrame with the new BDT score column added.
    """
    # 1️⃣ Load the BDT model
    with open(bdt_model_file, "rb") as f:
        bdt = pickle.load(f)

    # 2️⃣ Create a mask to remove NaNs and negative values
    mask = np.ones(len(evt_df), dtype=bool)
    for col in input_cols:
        mask &= evt_df[col].notna()
        mask &= evt_df[col] >= 0

    # 3️⃣ Build the input array
    X_BDT = evt_df.loc[mask, input_cols].values

    # 4️⃣ Evaluate the BDT score
    bdt_scores = bdt.decision_function(X_BDT)

    # 5️⃣ Add the score back to the DataFrame
    evt_df.loc[:,output_col] = -1.
    evt_df.loc[mask, output_col] = bdt_scores

    print(f"BDT scores added in column {output_col}.")
    print(f"Valid rows: {mask.sum()}, total rows: {len(evt_df)}")
    
    return evt_df


In [ ]:
# Color map
MC_COLORS = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#8c564b",  # brown
    "#e377c2",  # pink
    "#7f7f7f",  # gray
    "#bcbd22",  # olive
    "#17becf",  # cyan

    "#393b79",  # dark blue
    "#637939",  # dark green
    "#8c6d31",  # dark olive
    "#843c39",  # dark red
    "#7b4173",  # dark purple
    "#3182bd",  # steel blue
    "#31a354",  # teal green
    "#756bb1",  # muted violet
    "#636363",  # dark gray
    "#e6550d",  # burnt orange
]

In [ ]:
from dataclasses import dataclass, field
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import to_rgba

# --------------------------------------------------
# Config class (columns, bins, colors, labels)
# --------------------------------------------------
@dataclass
class HistogramConfig:
    data_column: tuple               # Column with the data to plot
    bins: np.ndarray = np.linspace(-0.5, 0.5, 51)
    color_map: dict = field(default_factory=dict)
    xlabel: str = 'Score'
    ylabel: str = 'Entries'
    title: str = None

# --------------------------------------------------
# Plotting function
# --------------------------------------------------
def plot_stacked_histogram(
    df,
    config: HistogramConfig,
    type_column: tuple,
    first_per_slice: bool = False,
):
    """
    Plot a stacked histogram using a HistogramConfig and an external type column.

    Parameters
    ----------
    df : pandas DataFrame
        The dataframe containing the data.
    config : HistogramConfig
        Configuration object with data_column, bins, colors, labels, and title.
    type_column : tuple
        Column used for stacking.
    first_per_slice : bool, optional
        If True, plot only the first PFP per slice
        (grouped by ['__ntuple', 'entry', 'rec.slc..index']).
        Default is False (use all rows).
    """

    slice_levels = ['__ntuple', 'entry', 'rec.slc..index']

    # Optionally reduce to first PFP per slice
    if first_per_slice:
        df = (
            df
            .groupby(level=slice_levels, sort=False)
            .first()
        )

    # Extract series
    data = df[config.data_column]
    types = df[type_column]

    # Unique type values (preserve order of appearance)
    type_values = list(types.dropna().unique())

    # Build stack data
    stack_data = {
        t: data[types == t].dropna()
        for t in type_values
    }

    # Colors (list-based, stable)
    colors = [MC_COLORS[i % len(MC_COLORS)] for i in range(len(type_values))]

    # Create figure
    fig, ax = plt.subplots()

    # Filled stack
    ax.hist(
        [stack_data[t] for t in type_values],
        bins=config.bins,
        stacked=True,
        histtype='stepfilled',
        color=colors,
        alpha=0.1,
        linewidth=0,
    )

    # Outline stack
    ax.hist(
        [stack_data[t] for t in type_values],
        bins=config.bins,
        stacked=True,
        histtype='step',
        color=colors,
        linewidth=2.0,
    )

    # Legend
    legend_handles = [
        Patch(
            facecolor=to_rgba(c, 0.1),
            edgecolor=to_rgba(c, 1.0),
            linewidth=2.0,
            label=t
        )
        for c, t in zip(colors, type_values)
    ]

    ax.set_xlabel(config.xlabel)
    ax.set_ylabel(config.ylabel)
    if config.title:
        ax.set_title(config.title)

    ax.legend(handles=legend_handles, loc='upper right')

    plt.show()

In [ ]:
BDT_input_columns = [
    ('pfp','trk','chi2pid','best','chi2_muon',''),
    ('pfp','trk','chi2pid','best','chi2_proton',''),
    ('pfp','trk','chi2_exp_pol','','',''),
    ('pfp','trk','frac50','','','')
]

col_BDT_score_proton = ('pfp','BDT_score_proton','','','','')

dev_sample_evt_df = add_bdt_score(
    dev_sample_evt_df,
    "bdt_model_proton.pkl",
    BDT_input_columns,
    col_BDT_score_proton
)


In [ ]:
cut_mask = CutMasks.nu_score_cut_mask(dev_sample_evt_df)  & CutMasks.track_cut_mask(dev_sample_evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(dev_sample_evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(dev_sample_evt_df,SLICE_LEVELS)

In [ ]:
plot_df = dev_sample_evt_df[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & (dev_sample_evt_df.pfp.is_exiting == False)]
config_BDT_proton_score = HistogramConfig(
    data_column=('pfp','BDT_score_proton','','','',''),
    bins=np.linspace(-1, 1.3, 51),
    xlabel='BDT proton score',
    ylabel='Entries',
    title='BDT proton Score'
)

plot_stacked_histogram(
    plot_df,
    config=config_BDT_proton_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)

# Training for µ/π separation

In [ ]:
chi2_p_cut = 80
chi2_mu_cut = 20
len_cut = 10
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')

col_chi2_exp_pol  = ('pfp', 'trk', 'chi2_exp_pol', '', '', '')
col_chi2_exp_pol_3var = ('pfp', 'trk', 'chi2_exp_pol_3var', '', '', '')
col_frac_50 = ('pfp', 'trk', 'frac50', '', '', '')
col_scatter_angle_ratio = ('pfp', 'scatter_angle_ratio', '', '', '', '')
col_max_daughter_hits = ('pfp', 'max_daughter_hits', '', '', '', '')
cut_mask = CutMasks.nu_score_cut_mask(evt_df)  & CutMasks.track_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.shower_cut_mask(evt_df, SLICE_LEVELS) & CutMasks.chi2_cut_mask(evt_df,SLICE_LEVELS)

#Define the track df
signal_df = evt_df[
    cut_mask &
    (abs(evt_df.pfp.trk.truth.p.pdg) == 13) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[
    cut_mask & 
    (abs(evt_df.pfp.trk.truth.p.pdg) == 211) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

#Not exiting for training
bkg_df[bkg_df.pfp.is_exiting == False]
signal_df[signal_df.pfp.is_exiting == False]

BDT_columns_mupi = [col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_scatter_angle_ratio, col_max_daughter_hits]
bdt_mask_signal = bdt_quality_mask(signal_df, BDT_columns_mupi)
bdt_mask_bkg    = bdt_quality_mask(bkg_df, BDT_columns_mupi)
signal_df = signal_df[bdt_mask_signal]
bkg_df    = bkg_df[bdt_mask_bkg]
signal_df = signal_df[(signal_df[col_chi2_mu] < chi2_mu_cut) & (signal_df[col_chi2_p] > chi2_p_cut) & (signal_df[col_len] > len_cut)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < chi2_mu_cut) & (bkg_df[col_chi2_p] > chi2_p_cut) & (bkg_df[col_len] > len_cut)]


In [ ]:
bkg_df.pfp.columns

In [ ]:

model_vec_mupi = create_models_for_columns(BDT_columns_mupi)
X_train_mupi, X_test_mupi, y_train_mupi, y_test_mupi = train_all_models(
    signal_df,
    bkg_df,
    BDT_columns_mupi,
    model_vec_mupi,
    test_size=0.2
)

In [ ]:
bdt_scores_mupi_train, bdt_scores_mupi_test   = get_scores(model_vec_mupi["BDT"], X_train_mupi, X_test_mupi)
bdtg_scores_mupi_train, bdtg_scores_mupi_test   = get_scores(model_vec_mupi["BDTG"], X_train_mupi, X_test_mupi)
bdtb_scores_mupi_train, bdtb_scores_mupi_test   = get_scores(model_vec_mupi["BDTB"], X_train_mupi, X_test_mupi)
rf_scores_mupi_train, rf_scores_mupi_test   = get_scores(model_vec_mupi["RF"], X_train_mupi, X_test_mupi)

plt.figure(figsize=(6,6))
plot_roc(y_test_mupi, bdt_scores_mupi_test, "BDT")
plot_roc(y_test_mupi, bdtg_scores_mupi_test, "BDTG")
plot_roc(y_test_mupi, bdtb_scores_mupi_test, "BDTB")
plot_roc(y_test_mupi, rf_scores_mupi_test, "RF")
plt.plot([0,1],[0,1], "k--")
plt.xlabel("Background efficiency")
plt.ylabel("Signal efficiency")
plt.legend()
plt.show()



sig_train = bdt_scores_mupi_train[y_train_mupi==1]
sig_test  = bdt_scores_mupi_test[y_test_mupi==1]
bkg_train = bdt_scores_mupi_train[y_train_mupi==0]
bkg_test  = bdt_scores_mupi_test[y_test_mupi==0]
plot_response(sig_train, sig_test, bkg_train, bkg_test, "BDT Response")


In [ ]:
with open("bdt_model_muon_pion.pkl", "wb") as f:
    pickle.dump(model_vec_mupi["BDT"], f)

In [ ]:
BDT_input_columns = [
    col_chi2_mu, col_chi2_p, col_chi2_exp_pol, col_scatter_angle_ratio, col_max_daughter_hits
]

col_BDT_score_muon_pion= ('pfp','BDT_score_muon_pion','','','','')

dev_sample_evt_df = add_bdt_score(
    dev_sample_evt_df,
    "bdt_model_muon_pion.pkl",
    BDT_input_columns,
    col_BDT_score_muon_pion
)

In [ ]:
plot_df = bdt_model_muon_pion[cut_mask & CutMasks.is_MIP_candidate_mask(dev_sample_evt_df) & (dev_sample_evt_df.pfp.is_exiting == False)]

config_BDT_muon_pion_score = HistogramConfig(
    data_column=('pfp','BDT_score_muon_pion','','','',''),
    bins=np.linspace(0.3, 0.75, 51),
    xlabel='BDT muon/pion score',
    ylabel='Entries',
    title='BDT muon/pion Score'
)

plot_stacked_histogram(
    plot_df,
    config=config_BDT_muon_pion_score,
    type_column=('pfp','trk','truth','p','p_type',''),
    first_per_slice = False
)